# Workshop 1: Sampling distributions (solutions)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture01/workshop01_solutions.ipynb)

In this workshop you write the code yourselves. Work in your group or alone, and ask the TAs when you are stuck.

Start with the core tasks on the RetailCorp data. Then do the short problems, which use small numbers and no
data, much like a question at the oral exam. The extension tasks are for those who have time left.

Each task ends with a question to answer in words. Under each task there are two hints: the first names the
statistical step, the second names the functions. Try the task before you open them. The slides and the
lecture notebook are your main references. You are welcome to use an LLM, but every task can be solved without
one, and you should be able to explain the statistics in your answers yourself.

This is the solutions version. Simulation results depend on the random draws, so your numbers will differ a
little from the ones here unless you use the same seeds.

## Toolbox

The functions you need today. The lecture notebook shows most of them in use.

| Function | What it does |
|---|---|
| `df["col"].mean()`, `.std()`, `.var()` | Mean, standard deviation and variance of a column (pandas divides by n − 1) |
| `np.std(x)`, `np.var(x)` | Standard deviation and variance (numpy divides by n unless you add `ddof=1`) |
| `df.groupby("col")["other"].agg(["mean", "std"])` | Statistics of one column for each group in another |
| `df[df["col"] == "value"]` | The rows where a column has a given value |
| `x.head(5)` | The first five values |
| `rng = np.random.default_rng(seed)` | A random number generator; the seed makes the results repeatable |
| `rng.choice(x, size=n, replace=False)` | A simple random sample of n values from x |
| `np.array(a_list)` | Turns a list into an array, so you can compute with it |
| `np.mean(a > value)` | The share of values in an array that are above a value |
| `np.sqrt(x)` | Square root |
| `stats.skew(x)` | Skewness |
| `stats.norm.cdf(x, loc, scale)`, `.sf`, `.ppf` | Normal distribution: probability below x, probability above x, value with a given probability below it |
| `stats.chi2.cdf(x, df)`, `.sf`, `.ppf` | The same for the chi-square distribution |
| `stats.binom.sf(k, n, p)` | Probability of more than k successes in n trials |
| `plt.hist(x, bins=30)` | Histogram |

For simulations, use the loop from the exercise cells in the lecture notebook: draw a sample inside a `for` loop,
compute a statistic, and append it to a list. Copy one of those cells and adapt it.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
sample = pd.read_csv(url + "orders_sample.csv")          # our random sample of 200 orders
population = pd.read_csv(url + "orders_population.csv")  # all 50,000 orders

## Core tasks

### Task 1. Describe the sample

Compute the mean and standard deviation of `delivery_hours` and the share of late orders in the sample. Then
compute the same three numbers for each warehouse.

**Answer in words.** Write two or three sentences for a manager at RetailCorp who has no training in statistics.

<details>
<summary>Hint 1</summary>

This is descriptive statistics on the sample. The share of late orders is the mean of the 0/1 column `late`.
</details>

<details>
<summary>Hint 2</summary>

`.mean()` and `.std()` on a column; `groupby("warehouse")` with `.agg(["mean", "std"])` for the groups.
</details>

In [ ]:
print(f"Mean delivery time:  {sample['delivery_hours'].mean():.2f} hours")
print(f"Standard deviation:  {sample['delivery_hours'].std():.2f} hours")
print(f"Share late:          {sample['late'].mean():.3f}")

In [ ]:
sample.groupby("warehouse")[["delivery_hours", "late"]].agg(["mean", "std"]).round(3)

**Answer.** In our sample of 200 orders, delivery took 36.8 hours on average, with a standard deviation of 8.1 hours, so
about two thirds of the orders arrived between roughly 29 and 45 hours. 8% of the orders took more than 48 hours.
Orders from Copenhagen took about 2.3 hours longer on average than orders from Aarhus (38.2 against 35.9 hours),
and a larger share of them were late (12.5% against 5.0%).

The standard deviation of `late` is not needed. Whether the warehouse difference is larger than chance can explain
is a question for Lecture 4; with 80 and 120 orders it may be.

### Task 2. Dividing by n or by n − 1

Compute the standard deviation and the variance of `delivery_hours` in the sample twice: with pandas and with
numpy (without `ddof`). Then do the same for only the first five orders in the sample.

**Answer in words.** Why do pandas and numpy give different results? Which of them gives an unbiased estimate of
the population variance σ²? How large is the difference for n = 200 and for n = 5, and what does that mean when
you work with small samples?

<details>
<summary>Hint 1</summary>

One of the two divides the sum of squared deviations by n, the other by n − 1. Compare the ratio of the two variances with (n − 1)/n.
</details>

<details>
<summary>Hint 2</summary>

`.std()` and `.var()` in pandas; `np.std(x)` and `np.var(x)` in numpy; `.head(5)` for the first five orders.
</details>

In [ ]:
x = sample["delivery_hours"]
print(f"n = 200: pandas sd {x.std():.3f}, numpy sd {np.std(x):.3f}")
print(f"         pandas var {x.var():.2f}, numpy var {np.var(x):.2f}, ratio {np.var(x) / x.var():.3f}")

x5 = x.head(5)
print(f"n = 5:   pandas sd {x5.std():.3f}, numpy sd {np.std(x5):.3f}")
print(f"         pandas var {x5.var():.2f}, numpy var {np.var(x5):.2f}, ratio {np.var(x5) / x5.var():.3f}")

**Answer.** pandas divides by n − 1 by default, numpy divides by n unless `ddof=1` is set. Dividing by n − 1 gives the
unbiased estimator s² of σ². The numpy variance is always (n − 1)/n times the pandas variance: 0.995 for n = 200,
where the difference does not matter in practice, and 0.8 for n = 5, where dividing by n understates the variance
by 20% on average. With small samples it matters which one you use, and you should check which convention a
function follows before you trust the number.

### Task 3. The sampling distribution of the mean basket value

Simulate 2,000 samples of n = 10 orders from the population and compute the mean basket value
(`order_value_dkk`) of each sample. Do the same for n = 50. Plot a histogram of the sample means for each n.

**Answer in words.** Compare the center, the spread and the shape of the two distributions with what theory
says. Use the population mean and standard deviation of basket values in your comparison.

<details>
<summary>Hint 1</summary>

Theory says the sample means are centered on μ, with standard deviation σ/√n, and approximately normal for large n. Compute μ and σ from the population first.
</details>

<details>
<summary>Hint 2</summary>

A `for` loop with `rng.choice(..., replace=False)` and `.mean()`, as in Exercise 2 of the lecture notebook; `np.std` on the array of means; `plt.hist`; `stats.skew` if you want a number for the shape.
</details>

In [ ]:
pop_values = population["order_value_dkk"].to_numpy()
mu_v, sigma_v = pop_values.mean(), pop_values.std()
print(f"Population: mean {mu_v:.0f} DKK, standard deviation {sigma_v:.0f} DKK")

rng = np.random.default_rng(11)
for n in (10, 50):
    means = []
    for _ in range(2000):
        x = rng.choice(pop_values, size=n, replace=False)
        means.append(x.mean())
    means = np.array(means)
    print(f"n = {n}: mean of means {means.mean():.0f}, sd of means {means.std():.0f}, "
          f"sigma/sqrt(n) {sigma_v / np.sqrt(n):.0f}, skewness {stats.skew(means):.2f}")
    plt.hist(means, bins=40, alpha=0.6, label=f"n = {n}")
plt.xlabel("Sample mean of basket value (DKK)")
plt.legend()
plt.show()

**Answer.** Both distributions are centered on the population mean of 593 DKK, so the sample mean is unbiased. The spread
matches σ/√n: about 165 DKK for n = 10 and about 74 DKK for n = 50 (the simulation gives values close to these).
The shape is still clearly skewed to the right for n = 10 and much closer to normal for n = 50, though a little
skew remains. Basket values are strongly skewed, so the central limit theorem needs a fairly large n here.

### Task 4. Probability for a sample mean

Use the population mean and standard deviation of `delivery_hours`. What is the probability that a random sample
of 50 orders has a mean delivery time above 38 hours? Then check your answer by simulation.

**Answer in words.** State the probability in a sentence, and say whether the simulation agrees.

<details>
<summary>Hint 1</summary>

Find the standard error σ/√n, standardize 38 hours, and find the probability above that z-value. For the check, simulate many sample means and count the share above 38.
</details>

<details>
<summary>Hint 2</summary>

`np.sqrt`, `stats.norm.sf`; for the simulation a `for` loop with `rng.choice` and then `np.mean(means > 38)`.
</details>

In [ ]:
hours = population["delivery_hours"].to_numpy()
mu, sigma = hours.mean(), hours.std()
n = 50
se = sigma / np.sqrt(n)
z = (38 - mu) / se
print(f"Standard error {se:.3f}, z = {z:.2f}, P(x_bar > 38) = {stats.norm.sf(z):.3f}")

rng = np.random.default_rng(13)
means = []
for _ in range(5000):
    x = rng.choice(hours, size=n, replace=False)
    means.append(x.mean())
means = np.array(means)
print(f"Simulated share of sample means above 38 hours: {np.mean(means > 38):.3f}")

**Answer.** The standard error is 7.82/√50 = 1.106 hours, so z = (38 − 36.40)/1.106 = 1.45 and the probability is about
0.074. About 7% of random samples of 50 orders would have a mean delivery time above 38 hours. The simulation
gives about the same share (0.076 with this seed).

### Task 5. Probability for a sample proportion

Consider only orders shipped from Copenhagen. What is the population share of late orders for Copenhagen? Check
whether the normal approximation is acceptable for samples of 100 Copenhagen orders, and use it to find the
probability that more than 15% of such a sample is late. Then check by simulation.

**Answer in words.** State the probability, compare it with the simulation, and explain why the two do not agree
exactly.

<details>
<summary>Hint 1</summary>

Select the Copenhagen orders, find P as the mean of `late`, check nP(1 − P) > 5, compute the standard error √(P(1 − P)/n), standardize 0.15 and find the probability above it. For the difference, think about which values p̂ can take with n = 100.
</details>

<details>
<summary>Hint 2</summary>

`population[population["warehouse"] == "Copenhagen"]`, `np.sqrt`, `stats.norm.sf`; a simulation loop with `rng.choice` on the Copenhagen `late` column. `stats.binom.sf` gives the exact probability if you want it.
</details>

In [ ]:
cph = population[population["warehouse"] == "Copenhagen"]["late"].to_numpy()
P_cph = cph.mean()
n = 100
print(f"Copenhagen orders: {len(cph):,}, share late P = {P_cph:.4f}")
print(f"nP(1 - P) = {n * P_cph * (1 - P_cph):.1f}")

se_p = np.sqrt(P_cph * (1 - P_cph) / n)
z = (0.15 - P_cph) / se_p
print(f"Standard error {se_p:.4f}, z = {z:.2f}, normal approximation {stats.norm.sf(z):.3f}")

rng = np.random.default_rng(14)
p_hats = []
for _ in range(5000):
    x = rng.choice(cph, size=n, replace=False)
    p_hats.append(x.mean())
p_hats = np.array(p_hats)
print(f"Simulated share of samples above 15% late: {np.mean(p_hats > 0.15):.3f}")
print(f"Exact binomial, at least 16 late of 100:   {stats.binom.sf(15, n, P_cph):.3f}")

**Answer.** 11.2% of Copenhagen orders are late. With n = 100, nP(1 − P) = 9.9, which is above 5, so the normal approximation
is acceptable by Newbold's rule. The standard error is 0.0315, z = 1.22, and the normal approximation gives a
probability of about 0.11. The simulation gives about 0.09, close to the exact binomial probability. The
difference arises because p̂ can only take the values 0, 0.01, 0.02 and so on: more than 15% late means at least
16 late orders, and the normal curve also counts the values between 0.15 and 0.16. With small n and a cutoff close
to a possible value of p̂, the approximation is rough even when the rule of thumb is met.

### Task 6. Probability for a sample variance

Use the population variance σ² of `delivery_hours`. What is the probability that the sample variance of 25
randomly chosen orders is more than 1.5 times σ²? Use the chi-square distribution, then check by simulation.
Finally, repeat the simulation for basket values (`order_value_dkk`), using their own population variance.

**Answer in words.** Does the simulation agree with the chi-square result for delivery times, and for basket
values? Explain the difference.

<details>
<summary>Hint 1</summary>

s² > 1.5σ² is the same as (n − 1)s²/σ² > (n − 1) × 1.5. Find the probability above that value in a chi-square distribution with n − 1 degrees of freedom. The chi-square result requires a normal population.
</details>

<details>
<summary>Hint 2</summary>

`stats.chi2.sf(value, df)`; a simulation loop with `rng.choice` and `.var(ddof=1)`, then `np.mean(...)` for the share above 1.5σ². `stats.skew` shows how far each variable is from normal.
</details>

In [ ]:
n = 25
print(f"Chi-square value: {(n - 1) * 1.5:.1f}, P = {stats.chi2.sf((n - 1) * 1.5, df=n - 1):.3f}")

rng = np.random.default_rng(15)
for variable in ("delivery_hours", "order_value_dkk"):
    pop = population[variable].to_numpy()
    sigma2 = pop.var()
    variances = []
    for _ in range(5000):
        x = rng.choice(pop, size=n, replace=False)
        variances.append(x.var(ddof=1))
    variances = np.array(variances)
    print(f"{variable}: skewness {stats.skew(pop):.2f}, "
          f"simulated share with s^2 > 1.5 sigma^2: {np.mean(variances > 1.5 * sigma2):.3f}")

**Answer.** The chi-square value is 24 × 1.5 = 36, and P(χ²₂₄ > 36) = 0.055. For delivery times the simulation gives about
the same, around 0.06, because delivery times are close to normal. For basket values the simulated share is about
three times as large (around 0.15), and the chi-square answer is wrong. Basket values are strongly skewed, and the
chi-square distribution of (n − 1)s²/σ² only holds for a normal population. Unlike for the mean, a larger sample
does not repair this.

## Short problems

These problems use small numbers and no data. Solve them with the formulas from the slides, and use Python only to
look up probabilities, or a table if you prefer. Write your reasoning in words, as you would explain it at the
exam.

### Problem 1. Single values and means

A machine fills coffee cups. The amount is normally distributed with mean 250 ml and standard deviation 10 ml.

1. What is the probability that a single cup contains less than 245 ml?
2. What is the probability that the mean of 16 cups is less than 245 ml?
3. Explain in words why the two answers are so different.

<details>
<summary>Hint 1</summary>

For one cup use σ; for the mean of 16 cups use the standard error σ/√16.
</details>

<details>
<summary>Hint 2</summary>

`stats.norm.cdf(245, loc=..., scale=...)`, once with scale 10 and once with the standard error.
</details>

In [ ]:
print(f"1. P(X < 245)     = {stats.norm.cdf(245, loc=250, scale=10):.4f}")
print(f"2. P(x_bar < 245) = {stats.norm.cdf(245, loc=250, scale=10 / np.sqrt(16)):.4f}")

**Answer.** 1. z = (245 − 250)/10 = −0.5, so P = 0.309. About 31% of single cups contain less than 245 ml.
2. The standard error is 10/√16 = 2.5 ml, so z = (245 − 250)/2.5 = −2 and P = 0.023.
3. Single cups vary with standard deviation 10 ml, but in a mean of 16 cups the high and low cups partly cancel
out, so the mean varies with only 2.5 ml. A mean far below 250 ml is therefore much less likely than a single
cup far below 250 ml.

### Problem 2. A sample proportion

Suppose 30% of the small and medium-sized firms in a region use AI tools. A researcher draws a random sample of
100 firms.

1. What are the mean and the standard error of the sample proportion p̂?
2. Is the normal approximation acceptable?
3. What is the probability that fewer than 25% of the firms in the sample use AI tools?
4. Would the normal approximation be acceptable with a sample of 20 firms?

<details>
<summary>Hint 1</summary>

E[p̂] = P and σ_p̂ = √(P(1 − P)/n). Newbold's rule of thumb is nP(1 − P) > 5.
</details>

<details>
<summary>Hint 2</summary>

`np.sqrt` for the standard error and `stats.norm.cdf` for the probability below 0.25.
</details>

In [ ]:
P, n = 0.30, 100
se_p = np.sqrt(P * (1 - P) / n)
print(f"1. mean {P}, standard error {se_p:.4f}")
print(f"2. nP(1 - P) = {n * P * (1 - P):.1f}")
print(f"3. P(p_hat < 0.25) = {stats.norm.cdf(0.25, loc=P, scale=se_p):.3f}")
print(f"4. with n = 20: nP(1 - P) = {20 * P * (1 - P):.1f}")

**Answer.** 1. The mean of p̂ is P = 0.30, and the standard error is √(0.30 × 0.70/100) = 0.046.
2. nP(1 − P) = 100 × 0.30 × 0.70 = 21, which is above 5, so the approximation is acceptable.
3. z = (0.25 − 0.30)/0.046 = −1.09, so the probability is about 0.14.
4. With n = 20, nP(1 − P) = 4.2, which is below 5. The sampling distribution of p̂ is then too lumpy and skewed
for the normal approximation to be trusted.

### Problem 3. A sample variance

The diameter of a machined part is normally distributed with standard deviation σ = 2 mm. A quality inspector
measures 10 parts and computes the sample variance s².

1. What is the probability that s² is larger than 6.5?
2. A colleague computes the variance of the 10 measurements by dividing by 10 instead of 9. On average, is the
result too large or too small, and by how much?
3. Why does the probability in question 1 require a normal population, while a probability for the mean of 10
parts would be less sensitive to that assumption?

<details>
<summary>Hint 1</summary>

(n − 1)s²/σ² follows a chi-square distribution with n − 1 degrees of freedom. Dividing by n instead of n − 1 multiplies the variance by (n − 1)/n.
</details>

<details>
<summary>Hint 2</summary>

`stats.chi2.sf(value, df=9)`.
</details>

In [ ]:
n, sigma2 = 10, 2**2
chi2_value = (n - 1) * 6.5 / sigma2
print(f"1. chi-square value {chi2_value:.3f}, P(s^2 > 6.5) = {stats.chi2.sf(chi2_value, df=n - 1):.3f}")
print(f"2. dividing by n gives on average (n - 1)/n = {(n - 1) / n} of sigma^2")

**Answer.** 1. σ² = 4, so χ² = 9 × 6.5/4 = 14.6 with 9 degrees of freedom, and P(χ²₉ > 14.6) = 0.10.
2. Too small. Dividing by n gives on average (n − 1)/n = 9/10 of σ², so the variance is understated by 10%.
3. The chi-square distribution of (n − 1)s²/σ² is derived for a normal population, and nothing like the central
limit theorem repairs it when the population is not normal (Task 6 shows how far off it can be). For the mean,
the central limit theorem makes x̄ approximately normal even when the population is not, as long as n is not too
small and the population not too skewed.

## Extension tasks

### Extension 1. The finite population correction

When the sample is a large part of the population, the standard error of x̄ is smaller than σ/√n (Newbold,
section 6.2). Compare two situations: a sample of 2,000 orders from all 50,000 orders, and a sample of 2,000 orders
from January only (about 7,800 orders). Compute the standard error with and without the finite population
correction √((N − n)/(N − 1)) in both situations, and check by simulation.

**Answer in words.** When does the correction matter?

<details>
<summary>Hint 1</summary>

Compute σ and N for each population. The corrected standard error is σ/√n × √((N − n)/(N − 1)). Simulate sample means without replacement and compare their standard deviation with both formulas.
</details>

<details>
<summary>Hint 2</summary>

Select January with `population[population["order_date"].str[:7] == "2026-01"]`; then `np.sqrt`, and a loop with `rng.choice(..., replace=False)`.
</details>

In [ ]:
january = population[population["order_date"].str[:7] == "2026-01"]["delivery_hours"].to_numpy()
all_orders = population["delivery_hours"].to_numpy()
n = 2000
rng = np.random.default_rng(17)
for label, pop in (("All orders", all_orders), ("January", january)):
    N, sigma = len(pop), pop.std()
    se = sigma / np.sqrt(n)
    se_fpc = se * np.sqrt((N - n) / (N - 1))
    means = []
    for _ in range(2000):
        x = rng.choice(pop, size=n, replace=False)
        means.append(x.mean())
    print(f"{label}: N = {N:,}, se {se:.3f}, with correction {se_fpc:.3f}, simulated {np.std(means):.3f}")

**Answer.** For all 50,000 orders the sample is 4% of the population, and the correction changes the standard error only
slightly (0.175 against 0.171). For January the sample is about a quarter of the population, and the correction
reduces the standard error from 0.177 to 0.153, which is what the simulation shows. The correction matters when the
sample is a sizable share of the population; Newbold mentions auditing as an example.

### Extension 2. Consistency of the sample variance

Simulate 1,000 samples of `delivery_hours` for n = 5, 20, 100 and 500, and compute the sample variance of each.
For each n, report the average and the standard deviation of the 1,000 sample variances.

**Answer in words.** What happens to the average and to the spread as n grows, and how does that relate to
unbiasedness and consistency?

<details>
<summary>Hint 1</summary>

Unbiased means the average equals σ² for every n. Consistent means the estimates concentrate around σ² as n grows, so the spread shrinks.
</details>

<details>
<summary>Hint 2</summary>

Two nested loops (one over n, one over the samples), `rng.choice` and `.var(ddof=1)`, then `np.mean` and `np.std`.
</details>

In [ ]:
hours = population["delivery_hours"].to_numpy()
print(f"Population variance: {hours.var():.1f}")
rng = np.random.default_rng(18)
for n in (5, 20, 100, 500):
    variances = []
    for _ in range(1000):
        x = rng.choice(hours, size=n, replace=False)
        variances.append(x.var(ddof=1))
    print(f"n = {n:>3}: average {np.mean(variances):.1f}, standard deviation {np.std(variances):.1f}")

**Answer.** The average stays close to the population variance of 61.1 for every n, because s² is unbiased. The standard
deviation of the estimates falls from about 46 for n = 5 to about 4 for n = 500, so the estimates concentrate
around σ² as n grows, which is what consistency means.